# 02 — Prepare Deterministic Train/Val/Test Split

This notebook generates `reports/split_indices.json` — the single source of
truth for the 80/10/10 train/val/test split used by all per-model training
notebooks (`02a`, `02b`, `02c`) and the evaluation notebook (`03`).

**Run this once before any training notebook.** Re-running with the same
dataset and seed produces identical output (idempotent).

Split parameters:
- Seed: `42`
- Ratios: `(0.8, 0.1, 0.1)` → train / val / test
- Method: `torch.utils.data.random_split` with `torch.Generator().manual_seed(42)`

## 1 · Environment detection

In [11]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

Running in Colab: True


## 2 · Mount Drive & set paths

In [12]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah')
    WORK_ROOT  = Path('/content/work')   # fast local SSD on Colab
else:
    # Local fallback — repo root is parent of scripts/
    DRIVE_ROOT = Path('..').resolve()
    WORK_ROOT  = DRIVE_ROOT

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'reports').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('WORK_ROOT  =', WORK_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT = /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah
WORK_ROOT  = /content/work


## 3 · Unzip / locate dataset

In [13]:
import zipfile
from pathlib import Path

DATA_ZIP  = DRIVE_ROOT / 'Dataset_Cropped.zip'
DATA_ROOT = None # Will be determined dynamically

# Check for existing dataset in common locations
if (WORK_ROOT / 'Dataset_Cropped' / 'defect').exists() and \
   (WORK_ROOT / 'Dataset_Cropped' / 'non_defect').exists():
    DATA_ROOT = WORK_ROOT / 'Dataset_Cropped'
    print(f'Dataset already present at {DATA_ROOT} — skipping extract.')
elif (WORK_ROOT / 'defect').exists() and \
     (WORK_ROOT / 'non_defect').exists():
    DATA_ROOT = WORK_ROOT
    print(f'Dataset already present at {DATA_ROOT} — skipping extract.')

if DATA_ROOT is None: # If not found, then extraction is needed
    if IN_COLAB:
        if not DATA_ZIP.exists():
            raise FileNotFoundError(
                f'Dataset zip not found.\n'
                f'Please upload Dataset_Cropped.zip to:\n  {DATA_ZIP}'
            )
        print(f'Extracting {DATA_ZIP} -> {WORK_ROOT} ...')
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(WORK_ROOT)
        print('Done.')

        # After extraction, try to determine DATA_ROOT again
        if (WORK_ROOT / 'Dataset_Cropped' / 'defect').exists() and \
           (WORK_ROOT / 'Dataset_Cropped' / 'non_defect').exists():
            DATA_ROOT = WORK_ROOT / 'Dataset_Cropped'
        elif (WORK_ROOT / 'defect').exists() and \
             (WORK_ROOT / 'non_defect').exists():
            DATA_ROOT = WORK_ROOT
        else:
            raise FileNotFoundError(
                f"Could not find 'Dataset_Cropped/defect' or 'defect' "
                f"folders under {WORK_ROOT} after extraction.\n"
                f"Please ensure the zip file contains 'defect/' and 'non_defect/' "
                f"either directly or nested within 'Dataset_Cropped/'."
            )
    else:
        raise FileNotFoundError(
            f'Dataset folder not found at:\n  {WORK_ROOT}/Dataset_Cropped or {WORK_ROOT}\n'
            f'Please run 01_auto_crop.py first to generate Dataset_Cropped/.'
        )

# Sanity check class folders
for c in ('defect', 'non_defect'):
    folder = DATA_ROOT / c
    if not folder.exists():
        # This part should ideally not be reached if DATA_ROOT was determined correctly above
        raise FileNotFoundError(
            f'Expected class folder not found: {folder}\n'
            f'Dataset_Cropped/ must contain defect/ and non_defect/ subfolders.'
            f'\nDetermined DATA_ROOT: {DATA_ROOT}'
        )
    n = len(list(folder.glob('*.jpg')))
    print(f'  {c:>10}: {n:>5} images')


Dataset already present at /content/work — skipping extract.
      defect:   500 images
  non_defect:   500 images


## 4 · Load dataset with ImageFolder

In [14]:
from torchvision import datasets

full_dataset = datasets.ImageFolder(root=str(DATA_ROOT), transform=None)
print('Classes (folder-alphabetic):', full_dataset.class_to_idx)
print('Total images:', len(full_dataset))

Classes (folder-alphabetic): {'defect': 0, 'non_defect': 1}
Total images: 1000


## 5 · Compute deterministic 80/10/10 split

In [15]:
import torch
from torch.utils.data import random_split

SEED = 42
SPLIT_RATIOS = (0.8, 0.1, 0.1)

n_total = len(full_dataset)
n_train = int(round(SPLIT_RATIOS[0] * n_total))
n_val   = int(round(SPLIT_RATIOS[1] * n_total))
n_test  = n_total - n_train - n_val

assert n_train + n_val + n_test == n_total, (
    f'Split sizes do not sum to total: {n_train}+{n_val}+{n_test} != {n_total}'
)

gen = torch.Generator().manual_seed(SEED)
train_subset, val_subset, test_subset = random_split(
    full_dataset, [n_train, n_val, n_test], generator=gen,
)

print(f'Split sizes — Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Sum check: {n_train + n_val + n_test} == {n_total} ✓')

Split sizes — Train: 800 | Val: 100 | Test: 100
Sum check: 1000 == 1000 ✓


## 6 · Write `reports/split_indices.json`

In [16]:
import json

split_payload = {
    'seed'         : SEED,
    'class_to_idx' : full_dataset.class_to_idx,
    'samples'      : [s for s, _ in full_dataset.samples],
    'train_indices': train_subset.indices,
    'val_indices'  : val_subset.indices,
    'test_indices' : test_subset.indices,
}

split_path = DRIVE_ROOT / 'reports' / 'split_indices.json'
split_path.write_text(json.dumps(split_payload, indent=2))
print(f'Saved -> {split_path}')
print(f'File size: {split_path.stat().st_size / 1024:.1f} KB')

Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/split_indices.json
File size: 55.7 KB


## 7 · Summary & sanity checks

In [17]:
import os

# Verify the JSON is readable and consistent
with open(split_path) as f:
    loaded = json.load(f)

assert loaded['seed'] == SEED
assert len(loaded['train_indices']) == n_train
assert len(loaded['val_indices'])   == n_val
assert len(loaded['test_indices'])  == n_test
assert len(loaded['samples'])       == n_total

# Check no index overlap
all_indices = set(loaded['train_indices']) | set(loaded['val_indices']) | set(loaded['test_indices'])
assert len(all_indices) == n_total, 'Index overlap detected!'
assert all_indices == set(range(n_total)), 'Indices do not cover full dataset!'

# Per-class distribution in each split
idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}
print('\n--- Split Summary ---')
print(f'  Total samples : {n_total}')
print(f'  Train         : {n_train} ({n_train/n_total*100:.1f}%)')
print(f'  Validation    : {n_val} ({n_val/n_total*100:.1f}%)')
print(f'  Test          : {n_test} ({n_test/n_total*100:.1f}%)')
print(f'  Seed          : {SEED}')
print(f'  No overlap    : ✓')
print(f'  Full coverage : ✓')

# Class distribution per split
print('\n--- Per-class distribution ---')
for split_name, indices in [('Train', loaded['train_indices']),
                             ('Val',   loaded['val_indices']),
                             ('Test',  loaded['test_indices'])]:
    counts = {}
    for idx in indices:
        _, label = full_dataset.samples[idx]
        cls_name = idx_to_class[label]
        counts[cls_name] = counts.get(cls_name, 0) + 1
    parts = ' | '.join(f'{k}: {v}' for k, v in sorted(counts.items()))
    print(f'  {split_name:>5}: {parts}')

print('\n✅ split_indices.json written successfully. '
      'Proceed to 02a/02b/02c training notebooks.')


--- Split Summary ---
  Total samples : 1000
  Train         : 800 (80.0%)
  Validation    : 100 (10.0%)
  Test          : 100 (10.0%)
  Seed          : 42
  No overlap    : ✓
  Full coverage : ✓

--- Per-class distribution ---
  Train: defect: 413 | non_defect: 387
    Val: defect: 39 | non_defect: 61
   Test: defect: 48 | non_defect: 52

✅ split_indices.json written successfully. Proceed to 02a/02b/02c training notebooks.
